# Filter SPICE Geometries for TMOS

This notebook uses the SPICE single-molecule subsets from the TMOS benchmark, but replaces the main assessment with a fast geometry-quality screen on the RDKit molecule generated from SPICE coordinates.

For each record, the workflow builds a mapped reference conformer from the SPICE mapped SMILES, then compares the SPICE-coordinate RDKit molecule against that reference using:

- 1-2 distance deviations
- 1-3 distance deviations
- nonbonded clash detection
- a hypervalent phosphorus heuristic

This is intended as a fast filter for unrealistic local geometries that could pollute force-field training.

## Prepare Geometry Screening

In [1]:
import hashlib
import json
import os
import re
import subprocess
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations

import h5py
import networkx as nx
import numpy as np
import periodictable as pt
import polars as pl
from openff.toolkit import Molecule
from openff.units import unit
from rdkit import Chem
from rdkit.Chem import Draw
import rdkit.Chem.rdDepictor as rdDepictor
import rdkit.Chem.rdDistGeom as rdDistGeom
import rdkit.Chem.rdForceFieldHelpers as rdForceFieldHelpers
import rdkit.Chem.rdchem as rdchem
import rdkit.Chem.rdmolops as rdmolops
from tqdm import tqdm

import tmos

SPICE_PATH = "/Users/jenniferclark/bin/back-to-school-jen/1_data/SPICE-2.0.1.hdf5"
BOHR_TO_ANGSTROM: float = (1.0 * unit.bohr).m_as(unit.angstrom)
PERIODIC_TABLE = getattr(Chem, "GetPeriodicTable")()

# Defaults are defined here so helper functions lint cleanly before the config cell runs.
BOND_DELTA_ABS_MAX_CUTOFF = 0.25
BOND_DELTA_ABS_MEAN_CUTOFF = 0.08
ONE_THREE_DELTA_ABS_MAX_CUTOFF = 0.40
ONE_THREE_DELTA_ABS_MEAN_CUTOFF = 0.12
NONBONDED_CLASH_SCALE = 0.75
NONBONDED_CLASH_COUNT_CUTOFF = 0
HYPERVALENT_P_COUNT_CUTOFF = 0


def sha256_file(file_path: str, chunk_size: int = 2**20) -> str:
    """Return SHA256 hex digest for a file."""
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def rdmol_positions(mol: rdchem.Mol) -> np.ndarray:
    """Return conformer positions as an array."""
    return np.asarray(mol.GetConformer().GetPositions(), dtype=float)


def reorder_mol_by_atom_map(mol: rdchem.Mol) -> tuple[rdchem.Mol, bool]:
    """Renumber a mapped molecule so atom index follows atom map order."""
    atom_map_numbers = [atom.GetAtomMapNum() for atom in mol.GetAtoms()]
    n_atoms = mol.GetNumAtoms()
    expected = list(range(1, n_atoms + 1))
    if sorted(atom_map_numbers) != expected:
        return rdchem.Mol(mol), False
    new_order = [
        idx for idx, _ in sorted(enumerate(atom_map_numbers), key=lambda item: item[1])
    ]
    return rdmolops.RenumberAtoms(mol, new_order), True


def build_reference_geometry(mapped_smiles: str) -> dict:
    """Build a mapped reference conformer and pair lists from SMILES."""
    offmol = Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True)
    reference_mol = offmol.to_rdkit()
    rdmolops.RemoveStereochemistry(reference_mol)
    reference_mol, used_atom_maps = reorder_mol_by_atom_map(reference_mol)

    params = rdDistGeom.ETKDGv3()
    params.randomSeed = 0xF00D
    status = rdDistGeom.EmbedMolecule(reference_mol, params)
    if status != 0:
        params.useRandomCoords = True
        status = rdDistGeom.EmbedMolecule(reference_mol, params)
    if status != 0:
        raise ValueError("RDKit could not embed a reference conformer from mapped SMILES")

    reference_method = "ETKDG"
    try:
        if rdForceFieldHelpers.MMFFHasAllMoleculeParams(reference_mol):
            rdForceFieldHelpers.MMFFOptimizeMolecule(reference_mol, maxIters=200)
            reference_method = "MMFF"
        else:
            rdForceFieldHelpers.UFFOptimizeMolecule(reference_mol, maxIters=200)
            reference_method = "UFF"
    except Exception:
        pass

    bond_pairs = sorted(
        tuple(sorted((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())))
        for bond in reference_mol.GetBonds()
    )

    one_three_pairs = set()
    for center in reference_mol.GetAtoms():
        neighbor_ids = [neighbor.GetIdx() for neighbor in center.GetNeighbors()]
        for idx_i, idx_j in combinations(neighbor_ids, 2):
            one_three_pairs.add(tuple(sorted((idx_i, idx_j))))

    return {
        "reference_mol": reference_mol,
        "reference_method": reference_method,
        "used_atom_maps": used_atom_maps,
        "reference_symbols": [atom.GetSymbol() for atom in reference_mol.GetAtoms()],
        "reference_positions": rdmol_positions(reference_mol),
        "bond_pairs": bond_pairs,
        "one_three_pairs": sorted(one_three_pairs),
        "topological_distances": rdmolops.GetDistanceMatrix(reference_mol),
    }


def build_spice_coordinate_mol(mapped_smiles: str, symbols: list[str], coordinates: np.ndarray) -> tuple[rdchem.Mol, int]:
    """Build the TMOS-style RDKit molecule from SPICE coordinates."""
    reference_graph = Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True).to_rdkit()
    rdmolops.RemoveStereochemistry(reference_graph)
    charge = sum(atom.GetFormalCharge() for atom in reference_graph.GetAtoms())
    spice_mol = tmos.build_rdmol.xyz_to_rdkit(symbols, coordinates, ignore_scale=True)
    rdmolops.RemoveStereochemistry(spice_mol)
    return spice_mol, charge


def check_graph_match(reference_mol: rdchem.Mol, spice_mol: rdchem.Mol) -> bool:
    """Check if the SPICE-coordinate mol graph is isomorphic to the reference SMILES graph."""
    nx_reference = tmos.graph_mapping.mol_to_graph(reference_mol)
    nx_spice = tmos.graph_mapping.mol_to_graph(spice_mol)
    node_match = nx.algorithms.isomorphism.categorical_node_match("symbol", None)
    gm = nx.algorithms.isomorphism.GraphMatcher(nx_reference, nx_spice, node_match=node_match)
    return gm.is_isomorphic()


def pair_distances(positions: np.ndarray, pairs: list[tuple[int, int]]) -> np.ndarray:
    """Compute Euclidean distances for indexed atom pairs."""
    if not pairs:
        return np.asarray([], dtype=float)
    return np.asarray([
        np.linalg.norm(positions[i] - positions[j])
        for i, j in pairs
    ], dtype=float)


def summarize_distance_deltas(reference_positions: np.ndarray, spice_positions: np.ndarray, pairs: list[tuple[int, int]]) -> dict:
    """Summarize absolute distance deltas for a collection of pairs."""
    if not pairs:
        return {"count": 0, "abs_mean": 0.0, "abs_max": 0.0, "worst_pair": None}
    reference_distances = pair_distances(reference_positions, pairs)
    spice_distances = pair_distances(spice_positions, pairs)
    absolute_deltas = np.abs(spice_distances - reference_distances)
    worst_index = int(np.argmax(absolute_deltas))
    return {
        "count": int(len(pairs)),
        "abs_mean": float(np.mean(absolute_deltas)),
        "abs_max": float(np.max(absolute_deltas)),
        "worst_pair": pairs[worst_index],
    }


def compute_nonbonded_clash_metrics(reference_mol: rdchem.Mol, spice_positions: np.ndarray, clash_scale: float) -> dict:
    """Measure nonbonded close contacts using reference graph exclusions."""
    topological_distances = rdmolops.GetDistanceMatrix(reference_mol)
    n_atoms = reference_mol.GetNumAtoms()
    clash_pairs = []
    min_gap = np.nan

    for atom_i in range(n_atoms):
        z_i = reference_mol.GetAtomWithIdx(atom_i).GetAtomicNum()
        radius_i = PERIODIC_TABLE.GetRvdw(z_i)
        for atom_j in range(atom_i + 1, n_atoms):
            if topological_distances[atom_i, atom_j] <= 2:
                continue
            z_j = reference_mol.GetAtomWithIdx(atom_j).GetAtomicNum()
            radius_j = PERIODIC_TABLE.GetRvdw(z_j)
            cutoff = clash_scale * (radius_i + radius_j)
            distance = float(np.linalg.norm(spice_positions[atom_i] - spice_positions[atom_j]))
            gap = distance - cutoff
            if np.isnan(min_gap) or gap < min_gap:
                min_gap = gap
            if gap < 0.0:
                clash_pairs.append((atom_i, atom_j, distance, cutoff, gap))

    clash_pairs.sort(key=lambda item: item[4])
    return {
        "count": int(len(clash_pairs)),
        "min_gap": float(min_gap) if not np.isnan(min_gap) else np.nan,
        "worst_pair": clash_pairs[0][:2] if clash_pairs else None,
        "pairs": clash_pairs,
    }


def geometry_flag_reasons(metrics: dict) -> list[str]:
    """Return threshold-based geometry flag reasons."""
    reasons = []
    if not metrics["graph_match"]:
        reasons.append("graph_mismatch")
    if metrics["bond_delta_abs_max"] > BOND_DELTA_ABS_MAX_CUTOFF:
        reasons.append("bond_delta_abs_max")
    if metrics["bond_delta_abs_mean"] > BOND_DELTA_ABS_MEAN_CUTOFF:
        reasons.append("bond_delta_abs_mean")
    if metrics["one_three_delta_abs_max"] > ONE_THREE_DELTA_ABS_MAX_CUTOFF:
        reasons.append("one_three_delta_abs_max")
    if metrics["one_three_delta_abs_mean"] > ONE_THREE_DELTA_ABS_MEAN_CUTOFF:
        reasons.append("one_three_delta_abs_mean")
    if metrics["nonbonded_clash_count"] > NONBONDED_CLASH_COUNT_CUTOFF:
        reasons.append("nonbonded_clash")
    if metrics["hypervalent_p_count"] > HYPERVALENT_P_COUNT_CUTOFF:
        reasons.append("hypervalent_p")
    return reasons


def assess_geometry_record(data: dict, reference_cache: dict[str, dict]) -> dict:
    """Assess one SPICE record against a mapped reference conformer."""
    mapped_smiles = data["smiles"]
    if mapped_smiles not in reference_cache:
        reference_cache[mapped_smiles] = build_reference_geometry(mapped_smiles)

    reference = reference_cache[mapped_smiles]
    spice_mol, charge = build_spice_coordinate_mol(mapped_smiles, data["symbols"], data["coords"][0])
    spice_positions = rdmol_positions(spice_mol)

    spice_symbols = [atom.GetSymbol() for atom in spice_mol.GetAtoms()]
    formula_match = Counter(spice_symbols) == Counter(reference["reference_symbols"])
    atom_order_matches_symbols = spice_symbols == reference["reference_symbols"]
    if not atom_order_matches_symbols:
        raise ValueError("Reference atom order does not match SPICE-coordinate atom order")

    graph_match = check_graph_match(reference["reference_mol"], spice_mol)

    bond_metrics = summarize_distance_deltas(
        reference["reference_positions"], spice_positions, reference["bond_pairs"],
    )
    one_three_metrics = summarize_distance_deltas(
        reference["reference_positions"], spice_positions, reference["one_three_pairs"],
    )
    clash_metrics = compute_nonbonded_clash_metrics(
        reference["reference_mol"], spice_positions, clash_scale=NONBONDED_CLASH_SCALE,
    )

    metrics = {
        "smiles": mapped_smiles,
        "symbols": data["symbols"],
        "coord": np.asarray(data["coords"][0]).tolist(),
        "charge": charge,
        "formula_match": formula_match,
        "atom_order_matches_symbols": atom_order_matches_symbols,
        "graph_match": graph_match,
        "reference_method": reference["reference_method"],
        "reference_used_atom_maps": reference["used_atom_maps"],
        "bond_pair_count": bond_metrics["count"],
        "bond_delta_abs_mean": bond_metrics["abs_mean"],
        "bond_delta_abs_max": bond_metrics["abs_max"],
        "bond_worst_pair": bond_metrics["worst_pair"],
        "one_three_pair_count": one_three_metrics["count"],
        "one_three_delta_abs_mean": one_three_metrics["abs_mean"],
        "one_three_delta_abs_max": one_three_metrics["abs_max"],
        "one_three_worst_pair": one_three_metrics["worst_pair"],
        "nonbonded_clash_count": clash_metrics["count"],
        "nonbonded_min_gap": clash_metrics["min_gap"],
        "nonbonded_worst_pair": clash_metrics["worst_pair"],
        "hypervalent_p_count": int(sum(
            atom.GetSymbol() == "P" and atom.GetDegree() >= 5
            for atom in spice_mol.GetAtoms()
        )),
    }

    reasons = geometry_flag_reasons(metrics)
    metrics["geometry_flag"] = bool(reasons)
    metrics["geometry_flag_reason"] = "|".join(reasons) if reasons else "pass"
    metrics["geometry_priority_score"] = float(
        metrics["bond_delta_abs_max"]
        + metrics["one_three_delta_abs_max"]
        + max(-metrics["nonbonded_min_gap"], 0.0)
        + float(metrics["hypervalent_p_count"])
    )
    return metrics


def pair_to_string(pair: tuple[int, int] | None) -> str | None:
    """Render an atom pair for summary tables."""
    if pair is None:
        return None
    return f"{pair[0]}-{pair[1]}"


/Users/jenniferclark/mamba/envs/qca/lib/python3.11/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
SPICE_URL = "https://zenodo.org/record/10975225/files/SPICE-2.0.1.hdf5?download=1"

if not os.path.exists(SPICE_PATH):
    data_dir = os.path.dirname(SPICE_PATH)
    output_file = SPICE_PATH
    commands = [
        f"mkdir -p {data_dir}",
        f"wget -O {output_file} {SPICE_URL}",
    ]
    for command in commands:
        subprocess.run(command, check=True, shell=True)

SPICE2_SOURCES: set[str] = {
    "SPICE DES Monomers Single Points Dataset v1.1",
    "SPICE Dipeptides Single Points Dataset v1.3",
    "SPICE PubChem Set 1 Single Points Dataset v1.3",
    "SPICE PubChem Set 2 Single Points Dataset v1.3",
    "SPICE PubChem Set 3 Single Points Dataset v1.3",
    "SPICE PubChem Set 4 Single Points Dataset v1.3",
    "SPICE PubChem Set 5 Single Points Dataset v1.3",
    "SPICE PubChem Set 6 Single Points Dataset v1.3",
    "SPICE PubChem Set 7 Single Points Dataset v1.0",
    "SPICE PubChem Set 8 Single Points Dataset v1.0",
    "SPICE PubChem Set 9 Single Points Dataset v1.0",
    "SPICE PubChem Set 10 Single Points Dataset v1.0",
}

# Get suspicious records from Lexie
with open("spice_upload_fails_atoms_too_close.json", "r") as f: 
    suspicious_set = set([x[0].split("-")[0] for ds, y in json.load(f).items() if ds in SPICE2_SOURCES for x in y])

BOND_DELTA_ABS_MAX_CUTOFF = 0.25
BOND_DELTA_ABS_MEAN_CUTOFF = 0.08
ONE_THREE_DELTA_ABS_MAX_CUTOFF = 0.40
ONE_THREE_DELTA_ABS_MEAN_CUTOFF = 0.12
NONBONDED_CLASH_SCALE = 0.5
NONBONDED_CLASH_COUNT_CUTOFF = 0
HYPERVALENT_P_COUNT_CUTOFF = 0

print("SPICE geometry filter thresholds:")
print(f"  bond max abs delta cutoff: {BOND_DELTA_ABS_MAX_CUTOFF:.2f} A")
print(f"  bond mean abs delta cutoff: {BOND_DELTA_ABS_MEAN_CUTOFF:.2f} A")
print(f"  1-3 max abs delta cutoff: {ONE_THREE_DELTA_ABS_MAX_CUTOFF:.2f} A")
print(f"  1-3 mean abs delta cutoff: {ONE_THREE_DELTA_ABS_MEAN_CUTOFF:.2f} A")
print(f"  nonbonded clash scale: {NONBONDED_CLASH_SCALE:.2f} x sum(vdW radii)")
print(f"  hypervalent P cutoff: > {HYPERVALENT_P_COUNT_CUTOFF}")

SPICE geometry filter thresholds:
  bond max abs delta cutoff: 0.25 A
  bond mean abs delta cutoff: 0.08 A
  1-3 max abs delta cutoff: 0.40 A
  1-3 mean abs delta cutoff: 0.12 A
  nonbonded clash scale: 0.50 x sum(vdW radii)
  hypervalent P cutoff: > 0


## Assess Geometry Quality

The denominator is all records from the selected SPICE subsets. Each record is screened on the RDKit molecule built from the SPICE coordinates.

In [3]:
TYPE_TO_SMARTS = {
    Chem.BondType.SINGLE: "-",
    Chem.BondType.DOUBLE: "=",
    Chem.BondType.TRIPLE: "#",
    Chem.BondType.AROMATIC: ":",
}


def _atom_descriptor_block(at_idx: int, mol: rdchem.Mol) -> str:
    atom = mol.GetAtomWithIdx(at_idx)
    charge = atom.GetFormalCharge()
    charge_str = f"+{charge}" if charge > 0 else str(charge)
    if charge == 0:
        charge_str = ""
    return f"#{atom.GetAtomicNum()}X{atom.GetDegree()}{charge_str}"


def _pair_to_string(pair: tuple[int, int] | list[int] | None) -> str | None:
    if pair is None:
        return None
    return f"{int(pair[0])}-{int(pair[1])}"


def _join_semicolon(values: list[str]) -> str:
    return ";".join(values)


def _recursive_atom_pattern(
    idx: int,
    mol: rdchem.Mol,
    recursion_level: int = 1,
    return_recursions: bool = False,
    include_bond_order: bool = True,
) -> str | list[str]:
    base = f"[{_atom_descriptor_block(idx, mol)}]"
    if recursion_level <= 0:
        return [] if return_recursions else base

    recursions = []
    for neighbor in mol.GetAtomWithIdx(idx).GetNeighbors():
        neighbor_idx = neighbor.GetIdx()
        neighbor_pattern = _recursive_atom_pattern(
            neighbor_idx,
            mol,
            recursion_level=recursion_level - 1,
            return_recursions=False,
            include_bond_order=include_bond_order,
        )
        if include_bond_order:
            bond = mol.GetBondBetweenAtoms(idx, neighbor_idx)
            bond_smarts = TYPE_TO_SMARTS.get(bond.GetBondType(), "~")
        else:
            bond_smarts = "~"
        recursions.append(f"{bond_smarts}{neighbor_pattern}")

    recursions.sort()
    if return_recursions:
        return recursions

    if recursions:
        return "[$(" + base + "".join(f"({r})" for r in recursions) + ")]"
    return base


def _mapped_atom_pattern(idx: int, map_id: int, mol: rdchem.Mol, recursion_level: int = 1) -> str:
    atom_base = _atom_descriptor_block(idx, mol)
    recursions = _recursive_atom_pattern(
        idx,
        mol,
        recursion_level=recursion_level,
        return_recursions=True,
        include_bond_order=True,
    )
    if recursions:
        rec_block = "".join(f"({r})" for r in recursions)
        return f"[{atom_base}&$([{atom_base}]{rec_block}):{map_id}]"
    return f"[{atom_base}:{map_id}]"


def _bond_smirks_with_context(mol: rdchem.Mol, atom_i: int, atom_j: int, recursion_level: int = 1) -> str:
    bond = mol.GetBondBetweenAtoms(atom_i, atom_j)
    if bond is None:
        return ""
    bond_smarts = TYPE_TO_SMARTS.get(bond.GetBondType(), "~")
    a1 = _mapped_atom_pattern(atom_i, 1, mol, recursion_level=recursion_level)
    a2 = _mapped_atom_pattern(atom_j, 2, mol, recursion_level=recursion_level)
    return f"{a1}{bond_smarts}{a2}"


def _angle_smirks_with_context(
    mol: rdchem.Mol,
    atom_i: int,
    atom_k: int,
    recursion_level: int = 1,
) -> list[str]:
    smirks = []
    neighbors_i = {n.GetIdx() for n in mol.GetAtomWithIdx(atom_i).GetNeighbors()}
    neighbors_k = {n.GetIdx() for n in mol.GetAtomWithIdx(atom_k).GetNeighbors()}
    centers = sorted(neighbors_i.intersection(neighbors_k))

    for center in centers:
        b1 = mol.GetBondBetweenAtoms(atom_i, center)
        b2 = mol.GetBondBetweenAtoms(center, atom_k)
        if b1 is None or b2 is None:
            continue
        b1s = TYPE_TO_SMARTS.get(b1.GetBondType(), "~")
        b2s = TYPE_TO_SMARTS.get(b2.GetBondType(), "~")

        a1 = _mapped_atom_pattern(atom_i, 1, mol, recursion_level=recursion_level)
        a2 = _mapped_atom_pattern(center, 2, mol, recursion_level=recursion_level)
        a3 = _mapped_atom_pattern(atom_k, 3, mol, recursion_level=recursion_level)
        smirks.append(f"{a1}{b1s}{a2}{b2s}{a3}")

    return smirks


def _problematic_pair_records(
    reference_positions: np.ndarray,
    spice_positions: np.ndarray,
    pairs: list[tuple[int, int]],
    cutoff: float,
) -> list[dict]:
    if not pairs:
        return []
    reference_distances = pair_distances(reference_positions, pairs)
    spice_distances = pair_distances(spice_positions, pairs)
    abs_deltas = np.abs(spice_distances - reference_distances)

    records = []
    for (atom_i, atom_j), abs_delta in zip(pairs, abs_deltas):
        if float(abs_delta) > float(cutoff):
            records.append({
                "pair": [int(atom_i), int(atom_j)],
                "abs_delta": float(abs_delta),
            })
    return records


def assess_geometry_record(data: dict, reference_cache: dict[str, dict]) -> dict:
    """Assess one SPICE record against a mapped reference conformer and catalog SMIRKS for problematic terms."""
    mapped_smiles = data["smiles"]
    if mapped_smiles not in reference_cache:
        reference_cache[mapped_smiles] = build_reference_geometry(mapped_smiles)

    reference = reference_cache[mapped_smiles]
    spice_mol, charge = build_spice_coordinate_mol(mapped_smiles, data["symbols"], data["coords"][0])
    spice_positions = rdmol_positions(spice_mol)

    spice_symbols = [atom.GetSymbol() for atom in spice_mol.GetAtoms()]
    formula_match = Counter(spice_symbols) == Counter(reference["reference_symbols"])
    atom_order_matches_symbols = spice_symbols == reference["reference_symbols"]
    if not atom_order_matches_symbols:
        raise ValueError("Reference atom order does not match SPICE-coordinate atom order")

    graph_match = check_graph_match(reference["reference_mol"], spice_mol)

    bond_metrics = summarize_distance_deltas(
        reference["reference_positions"], spice_positions, reference["bond_pairs"],
    )
    one_three_metrics = summarize_distance_deltas(
        reference["reference_positions"], spice_positions, reference["one_three_pairs"],
    )
    clash_metrics = compute_nonbonded_clash_metrics(
        reference["reference_mol"], spice_positions, clash_scale=NONBONDED_CLASH_SCALE,
    )

    problematic_bonds = _problematic_pair_records(
        reference["reference_positions"],
        spice_positions,
        reference["bond_pairs"],
        BOND_DELTA_ABS_MAX_CUTOFF,
    )
    problematic_angles = _problematic_pair_records(
        reference["reference_positions"],
        spice_positions,
        reference["one_three_pairs"],
        ONE_THREE_DELTA_ABS_MAX_CUTOFF,
    )

    for rec in problematic_bonds:
        atom_i, atom_j = rec["pair"]
        rec["smirks"] = _bond_smirks_with_context(spice_mol, atom_i, atom_j, recursion_level=1)

    for rec in problematic_angles:
        atom_i, atom_k = rec["pair"]
        rec["smirks"] = _angle_smirks_with_context(spice_mol, atom_i, atom_k, recursion_level=1)

    problematic_bond_smirks = sorted({
        rec["smirks"] for rec in problematic_bonds if rec["smirks"]
    })
    problematic_angle_smirks = sorted({
        sm for rec in problematic_angles for sm in rec["smirks"]
    })

    problematic_bond_pairs = sorted({
        _pair_to_string(rec["pair"]) for rec in problematic_bonds if _pair_to_string(rec["pair"]) is not None
    })
    problematic_angle_pairs = sorted({
        _pair_to_string(rec["pair"]) for rec in problematic_angles if _pair_to_string(rec["pair"]) is not None
    })

    metrics = {
        "smiles": mapped_smiles,
        "symbols": data["symbols"],
        "coord": np.asarray(data["coords"][0]).tolist(),
        "charge": charge,
        "formula_match": formula_match,
        "atom_order_matches_symbols": atom_order_matches_symbols,
        "graph_match": graph_match,
        "reference_method": reference["reference_method"],
        "reference_used_atom_maps": reference["used_atom_maps"],
        "bond_pair_count": bond_metrics["count"],
        "bond_delta_abs_mean": bond_metrics["abs_mean"],
        "bond_delta_abs_max": bond_metrics["abs_max"],
        "bond_worst_pair": _pair_to_string(bond_metrics["worst_pair"]),
        "one_three_pair_count": one_three_metrics["count"],
        "one_three_delta_abs_mean": one_three_metrics["abs_mean"],
        "one_three_delta_abs_max": one_three_metrics["abs_max"],
        "one_three_worst_pair": _pair_to_string(one_three_metrics["worst_pair"]),
        "nonbonded_clash_count": clash_metrics["count"],
        "nonbonded_min_gap": clash_metrics["min_gap"],
        "nonbonded_worst_pair": _pair_to_string(clash_metrics["worst_pair"]),
        "hypervalent_p_count": int(sum(
            atom.GetSymbol() == "P" and atom.GetDegree() >= 5
            for atom in spice_mol.GetAtoms()
        )),
        "problematic_bond_count": int(len(problematic_bonds)),
        "problematic_angle_count": int(len(problematic_angles)),
        "problematic_bond_pairs": _join_semicolon(problematic_bond_pairs),
        "problematic_angle_pairs": _join_semicolon(problematic_angle_pairs),
        "problematic_bond_smirks": _join_semicolon(problematic_bond_smirks),
        "problematic_angle_smirks": _join_semicolon(problematic_angle_smirks),
        "problematic_bonds": json.dumps(problematic_bonds),
        "problematic_angles": json.dumps(problematic_angles),
    }

    reasons = geometry_flag_reasons(metrics)
    metrics["geometry_flag"] = bool(reasons)
    metrics["geometry_flag_reason"] = "|".join(reasons) if reasons else "pass"
    metrics["geometry_priority_score"] = float(
        metrics["bond_delta_abs_max"]
        + metrics["one_three_delta_abs_max"]
        + max(-metrics["nonbonded_min_gap"], 0.0)
        + float(metrics["hypervalent_p_count"])
    )
    return metrics


print("Loaded recursion-1 explicit-bond SMIRKS cataloging with semicolon-delimited string fields.")

Loaded recursion-1 explicit-bond SMIRKS cataloging with semicolon-delimited string fields.


In [4]:
with h5py.File(SPICE_PATH) as spice:
    errors = defaultdict(list)
    outputs = []
    reference_cache = {}
    total_subset_records = 0
    n_structures_spice = 0
    n_structures_spice_subset = 0

    run_metadata = {
        "spice_path": SPICE_PATH,
        "spice_url": SPICE_URL,
        "spice_sha256": sha256_file(SPICE_PATH),
        "subsets": sorted(SPICE2_SOURCES),
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "reference_conformer_method": "ETKDG with MMFF or UFF relaxation when available",
        "bond_delta_abs_max_cutoff": BOND_DELTA_ABS_MAX_CUTOFF,
        "bond_delta_abs_mean_cutoff": BOND_DELTA_ABS_MEAN_CUTOFF,
        "one_three_delta_abs_max_cutoff": ONE_THREE_DELTA_ABS_MAX_CUTOFF,
        "one_three_delta_abs_mean_cutoff": ONE_THREE_DELTA_ABS_MEAN_CUTOFF,
        "nonbonded_clash_scale": NONBONDED_CLASH_SCALE,
        "hypervalent_p_count_cutoff": HYPERVALENT_P_COUNT_CUTOFF,
    }

    for key, record in tqdm(spice.items(), desc="Screening geometry", ncols=80):
        subset = record["subset"].asstr()[0]
        n_structures_spice += record["conformations"].shape[0]
        if subset not in SPICE2_SOURCES:
            continue
        n_structures_spice_subset += record["conformations"].shape[0]
        total_subset_records += 1

        smiles = record["smiles"].asstr()[0]
        atomic_number_dataset = record["atomic_numbers"]
        if atomic_number_dataset.dtype.kind in ("i", "u"):
            atomic_numbers = atomic_number_dataset[()].tolist()
        else:
            raw_atomic_numbers = atomic_number_dataset.asstr()[0]
            atomic_numbers = [int(x) for x in re.findall(r"-?\d+", raw_atomic_numbers)]

        atomic_numbers = [int(x) for x in atomic_numbers]
        elements = [pt.elements[z].symbol for z in atomic_numbers]

        n_conformers = record["conformations"].shape[0]
        assert len(record["dft_total_energy"]) == n_conformers
        coordinates = np.asarray([
            record["conformations"][i] * BOHR_TO_ANGSTROM
            for i in range(n_conformers)
        ], dtype=float)
        total_energy = record["dft_total_energy"][()].astype(float)
        data = {
            "smiles": smiles,
            "symbols": elements,
            "coords": coordinates,
            "total_energy": total_energy,
        }

        try:
            output = assess_geometry_record(data, reference_cache)
        except Exception as exc:
            err_key = (type(exc).__name__, str(exc))
            errors[err_key].append({
                "key": key,
                "data": data,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            })
        else:
            output["key"] = key
            output["subset"] = subset
            outputs.append(output)

geometry_df = pl.DataFrame(outputs, infer_schema_length=None)
total_errors = sum(len(values) for values in errors.values())
successful_records = int(geometry_df.height)

if total_subset_records != (successful_records + total_errors):
    raise RuntimeError(
        "Internal accounting mismatch: total_subset_records != successful_records + total_errors"
    )

run_metadata.update({
    "total_subset_records": int(total_subset_records),
    "successful_records": int(successful_records),
    "error_records": int(total_errors),
    "unique_reference_smiles": int(len(reference_cache)),
})

print(f"All of SPICE2: {n_structures_spice} conformers; Single molecule subset {n_structures_spice_subset} conformers")
print(
    f"Records in selected SPICE subsets: {total_subset_records}; successful: {successful_records}; exceptions: {total_errors}"
)

Screening geometry: 100%|█████████████| 113986/113986 [1:32:26<00:00, 20.55it/s]


All of SPICE2: 2008126 conformers; Single molecule subset 1279815 conformers
Records in selected SPICE subsets: 25607; successful: 25588; exceptions: 19


In [5]:
import h5py
import numpy as np
import nglview as nv
import ipywidgets as widgets
from IPython.display import display
from rdkit import Chem

TARGET_KEY = "135065494"

with h5py.File(SPICE_PATH, "r") as spice:
    if TARGET_KEY not in spice:
        raise KeyError(f"Record {TARGET_KEY} not found in SPICE file")

    record = spice[TARGET_KEY]
    mapped_smiles = record["smiles"].asstr()[0]

    atomic_number_dataset = record["atomic_numbers"]
    if atomic_number_dataset.dtype.kind in ("i", "u"):
        atomic_numbers = [int(x) for x in atomic_number_dataset[()]]
    else:
        raw_atomic_numbers = atomic_number_dataset.asstr()[0]
        atomic_numbers = [int(x) for x in re.findall(r"-?\d+", raw_atomic_numbers)]

    symbols = [pt.elements[z].symbol for z in atomic_numbers]
    conformations = np.asarray(record["conformations"][()], dtype=float) * BOHR_TO_ANGSTROM

base_mol, _ = build_spice_coordinate_mol(mapped_smiles, symbols, conformations[0])
base_mol.RemoveAllConformers()

for conf_index, coords in enumerate(conformations):
    conformer = Chem.Conformer(base_mol.GetNumAtoms())
    conformer.SetId(conf_index)
    for atom_idx, xyz in enumerate(coords):
        conformer.SetAtomPosition(atom_idx, (float(xyz[0]), float(xyz[1]), float(xyz[2])))
    base_mol.AddConformer(conformer, assignId=True)

n_conformers = base_mol.GetNumConformers()
current_idx = widgets.BoundedIntText(value=1, min=1, max=n_conformers, description="Conformer", disabled=True)
prev_button = widgets.Button(description="←", tooltip="Previous conformer")
next_button = widgets.Button(description="→", tooltip="Next conformer")
view_out = widgets.Output()
current_frame = {"value": 0}


def _render_frame(frame_idx: int) -> None:
    frame_idx = frame_idx % n_conformers
    current_frame["value"] = frame_idx
    current_idx.value = frame_idx + 1

    mol_one = Chem.Mol(base_mol)
    mol_one.RemoveAllConformers()
    mol_one.AddConformer(base_mol.GetConformer(frame_idx), assignId=True)

    with view_out:
        view_out.clear_output(wait=True)
        view = nv.show_rdkit(mol_one)
        view.clear_representations()
        view.add_ball_and_stick(multipleBond=True)
        view.layout.width = "100%"
        view.layout.height = "700px"
        display(view)


def _on_prev_click(_):
    _render_frame(current_frame["value"] - 1)


def _on_next_click(_):
    _render_frame(current_frame["value"] + 1)


prev_button.on_click(_on_prev_click)
next_button.on_click(_on_next_click)

controls = widgets.HBox([prev_button, current_idx, next_button])
print(f"Record {TARGET_KEY}: {n_conformers} conformers")
display(controls)
display(view_out)
_render_frame(0)

Record 135065494: 50 conformers


Output()

## Summarize the Filter

In [6]:
def _pct_string(n: int, denom: int) -> str:
    if denom <= 0:
        return "0.0000%"
    pct = n / denom * 100.0
    if pct < 0.01:
        return f"{pct:.4f}%"
    if pct < 0.1:
        return f"{pct:.3f}%"
    return f"{pct:.2f}%"

if geometry_df.is_empty():
    print("geometry_df is empty.")
else:
    denom_all = int(total_subset_records)
    success_n = int(geometry_df.height)
    error_n = int(sum(len(values) for values in errors.values()))
    flagged_n = int(geometry_df.filter(pl.col("geometry_flag")).height)

    print("Statistics denominator policy: all records in listed SPICE subsets.")
    print(
        f"Denominator (all subset records): {denom_all}; successful records: {success_n}; exceptions: {error_n}"
    )
    print(
        f"Flagged geometries: {flagged_n} / {denom_all} ({_pct_string(flagged_n, denom_all)})"
    )

    for col_name in [
        "formula_match",
        "atom_order_matches_symbols",
        "reference_used_atom_maps",
    ]:
        false_n = int(geometry_df.filter(pl.col(col_name) == False).height)
        print(f"{col_name} == False: {false_n} / {denom_all} ({_pct_string(false_n, denom_all)})")

    flagged_df = geometry_df.filter(pl.col("geometry_flag"))
    reason_counter = Counter()
    for reason_string in flagged_df.get_column("geometry_flag_reason").to_list():
        for reason in reason_string.split("|"):
            if reason and reason != "pass":
                reason_counter[reason] += 1

    print("\nFlag reasons:")
    for reason, count in reason_counter.most_common():
        print(f"  {reason}: {count} / {denom_all} ({_pct_string(count, denom_all)})")

    summary_columns = [
        "key",
        "subset",
        "geometry_flag_reason",
        "bond_delta_abs_max",
        "one_three_delta_abs_max",
        "nonbonded_clash_count",
        "nonbonded_min_gap",
        "hypervalent_p_count",
        "geometry_priority_score",
    ]
    print("\nHighest-priority flagged records:")
    print(
        flagged_df
        .sort("geometry_priority_score", descending=True)
        .select(summary_columns)
        .head(20)
    )

Statistics denominator policy: all records in listed SPICE subsets.
Denominator (all subset records): 25607; successful records: 25588; exceptions: 19
Flagged geometries: 629 / 25607 (2.46%)
formula_match == False: 0 / 25607 (0.0000%)
atom_order_matches_symbols == False: 0 / 25607 (0.0000%)
reference_used_atom_maps == False: 25588 / 25607 (99.93%)

Flag reasons:
  one_three_delta_abs_max: 404 / 25607 (1.58%)
  one_three_delta_abs_mean: 172 / 25607 (0.67%)
  graph_mismatch: 148 / 25607 (0.58%)
  nonbonded_clash: 86 / 25607 (0.34%)
  bond_delta_abs_max: 75 / 25607 (0.29%)
  bond_delta_abs_mean: 55 / 25607 (0.21%)
  hypervalent_p: 46 / 25607 (0.18%)

Highest-priority flagged records:
shape: (20, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ key       ┆ subset    ┆ geometry_ ┆ bond_delt ┆ … ┆ nonbonded ┆ nonbonded ┆ hypervale ┆ geometry │
│ ---       ┆ ---       ┆ flag_reas ┆ a_abs_max ┆   ┆ _clash_co ┆ _min_gap  ┆ nt_p_coun ┆ _pr

## Export and Inspect Flagged Cases

In [7]:
flagged_df = geometry_df.filter(pl.col("geometry_flag")).sort("geometry_priority_score", descending=True)

flagged_inputs = []
for row in flagged_df.to_dicts():
    flagged_inputs.append({
        "key": row["key"],
        "smiles": row["smiles"],
        "symbols": row["symbols"],
        "coordinates": row["coord"],
        "charge": row["charge"],
        "geometry_flag_reason": row["geometry_flag_reason"],
        "bond_delta_abs_max": row["bond_delta_abs_max"],
        "one_three_delta_abs_max": row["one_three_delta_abs_max"],
        "nonbonded_clash_count": row["nonbonded_clash_count"],
        "nonbonded_min_gap": row["nonbonded_min_gap"],
        "hypervalent_p_count": row["hypervalent_p_count"],
    })

with open("output_geometry_flagged.json", "w") as handle:
    json.dump(flagged_inputs, handle, indent=2)

print(f"Wrote {len(flagged_inputs)} flagged examples to output_geometry_flagged.json")
print(flagged_df.select(["key", "geometry_flag_reason"]).head(20))

Wrote 629 flagged examples to output_geometry_flagged.json
shape: (20, 2)
┌───────────┬─────────────────────────────────┐
│ key       ┆ geometry_flag_reason            │
│ ---       ┆ ---                             │
│ str       ┆ str                             │
╞═══════════╪═════════════════════════════════╡
│ 104040973 ┆ bond_delta_abs_mean|one_three_… │
│ 134973820 ┆ bond_delta_abs_mean|one_three_… │
│ 134976831 ┆ bond_delta_abs_mean|one_three_… │
│ 134983539 ┆ bond_delta_abs_mean|one_three_… │
│ 134987884 ┆ one_three_delta_abs_max|one_th… │
│ …         ┆ …                               │
│ 135092884 ┆ bond_delta_abs_mean             │
│ 135093775 ┆ bond_delta_abs_max|bond_delta_… │
│ 135095493 ┆ one_three_delta_abs_max|one_th… │
│ 135095649 ┆ bond_delta_abs_mean|one_three_… │
│ 135095949 ┆ one_three_delta_abs_max|one_th… │
└───────────┴─────────────────────────────────┘


In [8]:
# Compare current flagged records vs historical skip list and report stats
flagged = set(flagged_df["key"].to_list())

overlap = flagged & suspicious_set
new_problematic = flagged - suspicious_set
skipped_not_flagged_now = suspicious_set - flagged

def _pct(n: int, d: int) -> float:
    return (100.0 * n / d) if d else 0.0

print("Comparison: current flagged vs skip_records")
print(f"  flagged now:                 {len(flagged)}")
print(f"  in skip_records:             {len(suspicious_set)}")
print(f"  overlap:                     {len(overlap)} ({_pct(len(overlap), len(flagged)):.2f}% of flagged)")
print(f"  NEW problematic (flagged-skip): {len(new_problematic)} ({_pct(len(new_problematic), len(flagged)):.2f}% of flagged)")
print(f"  skipped but not flagged now: {len(skipped_not_flagged_now)} ({_pct(len(skipped_not_flagged_now), len(suspicious_set)):.2f}% of skip_records)")

stats_df = pl.DataFrame({
    "metric": [
        "flagged_now",
        "skip_records",
        "overlap",
        "new_problematic",
        "skipped_not_flagged_now",
    ],
    "count": [
        len(flagged),
        len(suspicious_set),
        len(overlap),
        len(new_problematic),
        len(skipped_not_flagged_now),
    ],
})
stats_df

Comparison: current flagged vs skip_records
  flagged now:                 629
  in skip_records:             123
  overlap:                     85 (13.51% of flagged)
  NEW problematic (flagged-skip): 544 (86.49% of flagged)
  skipped but not flagged now: 38 (30.89% of skip_records)


metric,count
str,i64
"""flagged_now""",629
"""skip_records""",123
"""overlap""",85
"""new_problematic""",544
"""skipped_not_flagged_now""",38


In [9]:
def detailed_pair_delta_table(case: dict, reference: dict) -> pl.DataFrame:
    reference_positions = reference["reference_positions"]
    spice_positions = np.asarray(case["coord"], dtype=float)

    rows = []
    for label, pairs in [("bond", reference["bond_pairs"]), ("one_three", reference["one_three_pairs"])]:
        reference_distances = pair_distances(reference_positions, pairs)
        spice_distances = pair_distances(spice_positions, pairs)
        for (atom_i, atom_j), ref_distance, spice_distance in zip(pairs, reference_distances, spice_distances):
            rows.append({
                "pair_type": label,
                "pair": f"{atom_i}-{atom_j}",
                "reference_distance": float(ref_distance),
                "spice_distance": float(spice_distance),
                "abs_delta": float(abs(spice_distance - ref_distance)),
            })
    return pl.DataFrame(rows).sort("abs_delta", descending=True)

In [10]:
flagged_df_p = flagged_df.filter(pl.col("symbols").list.contains("P"))
flagged_df_p

smiles,symbols,coord,charge,formula_match,atom_order_matches_symbols,graph_match,reference_method,reference_used_atom_maps,bond_pair_count,bond_delta_abs_mean,bond_delta_abs_max,bond_worst_pair,one_three_pair_count,one_three_delta_abs_mean,one_three_delta_abs_max,one_three_worst_pair,nonbonded_clash_count,nonbonded_min_gap,nonbonded_worst_pair,hypervalent_p_count,problematic_bond_count,problematic_angle_count,problematic_bond_pairs,problematic_angle_pairs,problematic_bond_smirks,problematic_angle_smirks,problematic_bonds,problematic_angles,geometry_flag,geometry_flag_reason,geometry_priority_score,key,subset
str,list[str],list[list[f64]],i64,bool,bool,bool,str,bool,i64,f64,f64,str,i64,f64,f64,str,i64,f64,str,i64,i64,i64,str,str,str,str,str,str,bool,str,f64,str,str
"""[F:1][P:6]([F:2])([F:3])([F:4]…","[""F"", ""F"", … ""P""]","[[-0.310209, -0.300876, -0.619075], [0.690374, 1.708376, -0.543133], … [0.566218, 0.557118, 0.55668]]",0,true,true,true,"""UFF""",false,5,0.045921,0.1041,"""1-5""",10,0.561087,1.523869,"""3-4""",0,NaN,null,1,0,5,"""""","""0-3;0-4;1-2;1-4;3-4""","""""","""[#9X1&$([#9X1](-[#15X5])):1]-[…","""[]""","""[{""pair"": [0, 3], ""abs_delta"":…",true,"""one_three_delta_abs_max|one_th…",NaN,"""134987884""","""SPICE PubChem Set 1 Single Poi…"
"""[O:1]=[P:5]([Br:2])([Br:3])[Br…","[""O"", ""Br"", … ""P""]","[[-0.819339, -0.451701, 0.495408], [1.141066, 0.522208, 2.646001], … [0.39087, 0.423434, 0.507855]]",0,true,true,true,"""MMFF""",false,4,0.079348,0.169692,"""3-4""",6,0.128334,0.313167,"""0-3""",0,NaN,null,0,0,0,"""""","""""","""""","""""","""[]""","""[]""",true,"""one_three_delta_abs_mean""",NaN,"""134989033""","""SPICE PubChem Set 6 Single Poi…"
"""[P:6]([Cl:1])([Cl:2])([Cl:3])(…","[""Cl"", ""Cl"", … ""P""]","[[0.21336, -1.288363, -0.293232], [-1.373063, 0.371744, 1.273195], … [0.576518, 0.540893, 0.574952]]",0,true,true,true,"""UFF""",false,5,0.110339,0.276956,"""2-5""",10,0.784752,1.69645,"""0-1""",0,NaN,null,1,1,8,"""2-5""","""0-1;0-2;0-3;0-4;1-2;1-3;1-4;3-…","""[#17X1&$([#17X1](-[#15X5])):1]…","""[#17X1&$([#17X1](-[#15X5])):1]…","""[{""pair"": [2, 5], ""abs_delta"":…","""[{""pair"": [0, 1], ""abs_delta"":…",true,"""bond_delta_abs_max|bond_delta_…",NaN,"""134990319""","""SPICE PubChem Set 6 Single Poi…"
"""[P:6]([Br:1])([Br:2])([Br:3])(…","[""Br"", ""Br"", … ""P""]","[[0.821502, 1.335091, -1.550026], [2.139574, -0.89696, 0.783781], … [0.549817, 0.627015, 0.47089]]",0,true,true,true,"""UFF""",false,5,0.102565,0.234611,"""0-5""",10,0.763763,2.421165,"""0-3""",0,NaN,null,1,0,5,"""""","""0-3;1-2;1-3;1-4;2-3""","""""","""[#35X1&$([#35X1](-[#15X5])):1]…","""[]""","""[{""pair"": [0, 3], ""abs_delta"":…",true,"""bond_delta_abs_mean|one_three_…",NaN,"""135020045""","""SPICE PubChem Set 1 Single Poi…"
"""[O-:1][P:4]([O-:2])[O-:3]""","[""O"", ""O"", … ""P""]","[[-0.249525, 1.578781, -0.101816], [1.273335, 0.834955, 1.834546], … [0.57588, 0.525416, 0.515237]]",-3,true,true,true,"""MMFF""",false,3,0.170456,0.194736,"""0-3""",3,0.074378,0.16314,"""1-2""",0,NaN,null,0,0,0,"""""","""""","""""","""""","""[]""","""[]""",true,"""bond_delta_abs_mean""",NaN,"""135064957""","""SPICE PubChem Set 1 Single Poi…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""[H:15][C:4]1=[C:6]([N:10]2[C:7…","[""O"", ""O"", … ""H""]","[[2.083004, 2.557938, 1.040742], [1.342274, 0.485584, 2.298664], … [1.079222, -2.969467, -0.277787]]",0,true,true,false,"""MMFF""",false,27,0.027627,0.080502,"""5-9""",47,0.070642,0.226998,"""2-9""",0,0.928463,null,0,0,0,"""""","""""","""""","""""","""[]""","""[]""",true,"""graph_mismatch""",0.3075,"""135239583""","""SPICE PubChem Set 4 Single Poi…"
"""[O:1]=[C:11]([O-:3])[C:13]1=[C…","[""O"", ""O"", … ""H""]","[[3.076438, -4.124663, 2.44904], [-1.813179, 3.916265, -2.843766], … [-0.602323, -0.142694, -2.058754]]",-3,true,true,true,"""MMFF""",false,34,0.034573,0.108019,"""18-19""",59,0.074186,0.175304,"""7-19""",1,-0.023607,"""3-22""",0,0,0,"""""","""""","""""","""""","""[]""","""[]""",true,"""nonbon

In [13]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import nglview as nv
import html
from rdkit.Chem import AllChem

class GeometryInspector:
    def __init__(self, flagged_df, outputs, reference_cache):
        self.flagged_df = flagged_df
        self.outputs = outputs
        self.reference_cache = reference_cache
        self.current_index = 0

        self.ok_geometry_keys = []
        self.not_ok_geometry_keys = []

        self.info_html = widgets.HTML(value="")
        self.view_container = widgets.VBox(children=[])

        self.prev_button = widgets.Button(description="← Previous", disabled=True)
        self.next_button = widgets.Button(description="Next →")
        self.ok_button = widgets.Button(description="✓ OK Geometry", button_style="success")
        self.not_ok_button = widgets.Button(description="✗ Not OK", button_style="danger")

        self.counter_label = widgets.Label(value="")
        self.classification_label = widgets.Label(value="")

        self.prev_button.on_click(self._on_prev_click)
        self.next_button.on_click(self._on_next_click)
        self.ok_button.on_click(self._on_ok_click)
        self.not_ok_button.on_click(self._on_not_ok_click)

    def _get_current_case(self):
        case_key = self.flagged_df.item(self.current_index, "key")
        case = next(row for row in self.outputs if row["key"] == case_key)
        reference = self.reference_cache[case["smiles"]]
        return case, reference, case_key

    def _on_prev_click(self, b):
        if self.current_index > 0:
            self.current_index -= 1
            self._update_display()

    def _on_next_click(self, b):
        if self.current_index < len(self.flagged_df) - 1:
            self.current_index += 1
            self._update_display()

    def _on_ok_click(self, b):
        _, _, case_key = self._get_current_case()
        if case_key not in self.ok_geometry_keys:
            self.ok_geometry_keys.append(case_key)
        if case_key in self.not_ok_geometry_keys:
            self.not_ok_geometry_keys.remove(case_key)
        self._on_next_click(None)

    def _on_not_ok_click(self, b):
        _, _, case_key = self._get_current_case()
        if case_key not in self.not_ok_geometry_keys:
            self.not_ok_geometry_keys.append(case_key)
        if case_key in self.ok_geometry_keys:
            self.ok_geometry_keys.remove(case_key)
        self._on_next_click(None)

    def _update_classification_label(self):
        _, _, case_key = self._get_current_case()
        if case_key in self.ok_geometry_keys:
            self.classification_label.value = "✓ Classified as OK"
        elif case_key in self.not_ok_geometry_keys:
            self.classification_label.value = "✗ Classified as NOT OK"
        else:
            self.classification_label.value = ""

    def _update_display(self):
        case, reference, case_key = self._get_current_case()

        # Build info text and set directly on HTML widget — no Output buffering
        lines = [
            f"Key: {html.escape(case_key)}",
            f"Subset: {html.escape(case['subset'])}",
            f"Reasons: {html.escape(case['geometry_flag_reason'])}",
            case['smiles'],
            "",
            html.escape(str(detailed_pair_delta_table(case, reference).head(5))),
        ]
        self.info_html.value = "<pre>" + "\n".join(lines) + "</pre>"

        # Build new 3D views and swap children atomically
        reference_mol = rdchem.Mol(reference["reference_mol"])
        reference_mol.AddConformer(rdchem.Conformer(reference_mol.GetNumAtoms()))
        AllChem.EmbedMolecule(reference_mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(reference_mol)
        spice_mol, _ = build_spice_coordinate_mol(
            case["smiles"], case["symbols"], np.asarray(case["coord"], dtype=float)
        )

        view_ref = nv.show_rdkit(reference_mol)
        view_spice = nv.show_rdkit(spice_mol)
        #view_ref.layout = widgets.Layout(width="500px", height="400px")
        #view_spice.layout = widgets.Layout(width="500px", height="400px")

        ref_box = widgets.VBox([
            widgets.Label("Reference conformer", layout=widgets.Layout(width="500px")),
            view_ref,
        ])
        spice_box = widgets.VBox([
            widgets.Label("SPICE coordinates", layout=widgets.Layout(width="500px")),
            view_spice,
        ])
        self.view_container.children = [widgets.HBox([ref_box, spice_box])]

        self.prev_button.disabled = (self.current_index == 0)
        self.next_button.disabled = (self.current_index == len(self.flagged_df) - 1)
        self.counter_label.value = f"Record {self.current_index + 1} of {len(self.flagged_df)}"
        self._update_classification_label()

    def show(self):
        nav_box = widgets.HBox([self.prev_button, self.counter_label, self.next_button])
        button_box = widgets.HBox([self.ok_button, self.not_ok_button])
        main_box = widgets.VBox([
            nav_box,
            self.info_html,
            self.view_container,
            self.classification_label,
            button_box,
        ])
        display(main_box)
        self._update_display()

# Initialize the inspector
inspector = GeometryInspector(
    flagged_df.filter(~pl.col("key").is_in(flagged_df_p["key"])),
    outputs,
    reference_cache,
)
inspector.show()


/var/folders/dr/9nnhm1493kv7_s9r_wj_l_jm0000gn/T/ipykernel_18939/2291312079.py:133: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  flagged_df.filter(~pl.col("key").is_in(flagged_df_p["key"])),


KekulizeException: Can't kekulize mol.  Unkekulized atoms: 0 3 4 5 6 7

In [14]:
# Export SMILES for skip_records and flagged_df
from pathlib import Path

skip_record_keys = sorted(str(key) for key in suspicious_set)

# Resolve skip-record SMILES by key from the SPICE file.
skip_smiles_rows = []
with h5py.File(SPICE_PATH, "r") as spice:
    for key in skip_record_keys:
        if key not in spice:
            continue
        smiles = spice[key]["smiles"].asstr()[0]
        skip_smiles_rows.append({"key": key, "smiles": smiles})

skip_smiles_df = pl.DataFrame(skip_smiles_rows)

# flagged_df already contains smiles.
if "key" in flagged_df.columns:
    flagged_smiles_df = (
        flagged_df
        .select(["key", "smiles"])
        .unique()
        .sort("key")
    )
else:
    flagged_smiles_df = (
        flagged_df
        .select(["smiles"])
        .unique()
        .sort("smiles")
    )

# Write outputs.
out_dir = Path(".")
out_skip_txt = out_dir / "skip_records_smiles.txt"
out_skip_csv = out_dir / "skip_records_smiles.csv"
out_flagged_txt = out_dir / "flagged_df_smiles.txt"
out_flagged_csv = out_dir / "flagged_df_smiles.csv"

if skip_smiles_df.height:
    out_skip_txt.write_text("\n".join(skip_smiles_df.get_column("smiles").to_list()) + "\n")
    skip_smiles_df.write_csv(out_skip_csv)
else:
    out_skip_txt.write_text("")
    pl.DataFrame({"key": [], "smiles": []}).write_csv(out_skip_csv)

if "smiles" in flagged_smiles_df.columns and flagged_smiles_df.height:
    out_flagged_txt.write_text("\n".join(flagged_smiles_df.get_column("smiles").to_list()) + "\n")
else:
    out_flagged_txt.write_text("")

flagged_smiles_df.write_csv(out_flagged_csv)

print(f"Wrote {skip_smiles_df.height} skip-record SMILES to {out_skip_txt} and {out_skip_csv}")
print(f"Wrote {flagged_smiles_df.height} flagged SMILES to {out_flagged_txt} and {out_flagged_csv}")

skip_smiles_df.head(), flagged_smiles_df.head()

Wrote 123 skip-record SMILES to skip_records_smiles.txt and skip_records_smiles.csv
Wrote 629 flagged SMILES to flagged_df_smiles.txt and flagged_df_smiles.csv


(shape: (5, 2)
 ┌───────────┬─────────────────────────────────┐
 │ key       ┆ smiles                          │
 │ ---       ┆ ---                             │
 │ str       ┆ str                             │
 ╞═══════════╪═════════════════════════════════╡
 │ 103929721 ┆ [O:1]=[P:17]([O:6][H:20])([O:7… │
 │ 103937430 ┆ [H:18][c:8]1[c:7]([c:9]([c:11]… │
 │ 103940402 ┆ [O:1]=[P:25]([O:4][H:27])([O:5… │
 │ 103940406 ┆ [H:24][C:7](=[C:6]([C:11]([H:2… │
 │ 103940422 ┆ [O:1]=[P:18]([O:3][H:20])([O:4… │
 └───────────┴─────────────────────────────────┘,
 shape: (5, 2)
 ┌───────────┬─────────────────────────────────┐
 │ key       ┆ smiles                          │
 │ ---       ┆ ---                             │
 │ str       ┆ str                             │
 ╞═══════════╪═════════════════════════════════╡
 │ 103915892 ┆ [H:27][c:5]1[c:8]([c:12]([c:16… │
 │ 103928820 ┆ [O:1]=[N+:21]([O-:4])[c:18]1[c… │
 │ 103929721 ┆ [O:1]=[P:17]([O:6][H:20])([O:7… │
 │ 103930354 ┆ [O:1]=[P:8]([O:3][H:10]